<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [ ]:
# Install spark

In [1]:
!pip install pyspark
!pip install findspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 10.1 MB/s  0:00:44:00:0100:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798675 sha256=39248f2f5513aa7d9592932c9b949eb16345fb0c217d573982a46bdca0bbe615
  Stored in directory: /Users/juanzinser/Library/Caches/pip/wheels/4f/6d/69/496f1f5964e3470c76313fe2a499eb219a9c95ac1f1cb880ab
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [5]:
# Start session
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [7]:
!export JAVA_HOME=$(/usr/libexec/java_home -v 17)

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



In [8]:
import os
result = os.popen('/usr/libexec/java_home -v 17').read().strip()
os.environ['JAVA_HOME'] = result
print(os.environ['JAVA_HOME'])

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



In [9]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Saving and Loading a SparkML Model").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/17 11:08:48 WARN Utils: Your hostname, Juans-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.100 instead (on interface en0)
26/09/17 11:08:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/juanzinser/Documents/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/17 11:08:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Download The search term dataset from the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [10]:
# Load the csv into a spark dataframe
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("searchterms.csv")

df.show(5)


+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows


In [11]:
# Print the number of rows and columns
num_rows = df.count()
num_cols = len(df.columns)

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")

Number of rows: 10000
Number of columns: 4


In [12]:
# Print the top 5 rows
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows


In [13]:
# Find out the datatype of the column searchterm?
df.schema["searchterm"].dataType

StringType()

In [14]:
# How many times was the term `gaming laptop` searched?
count = df.filter(df.searchterm == "gaming laptop").count()
print(f"'gaming laptop' was searched {count} times")

'gaming laptop' was searched 499 times


In [15]:
# Print the top 5 most frequently used search terms?
df.groupBy("searchterm") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(5)

+-------------+-----+
|   searchterm|count|
+-------------+-----+
|mobile 6 inch| 2312|
|    mobile 5g| 2301|
|mobile latest| 1327|
|       laptop|  935|
|  tablet wifi|  896|
+-------------+-----+
only showing top 5 rows


In [ ]:
# The pretrained sales forecasting model is available at  the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

In [17]:
# Load the sales forecast model.
from pyspark.ml.regression import LinearRegressionModel

model = LinearRegressionModel.load("sales_prediction.model")


In [19]:
from pyspark.ml.feature import VectorAssembler

# Using the sales forecast model, predict the sales for the year of 2023.
data = [(2023,)]
columns = ["year"]
new_df = spark.createDataFrame(data, columns)

assembler = VectorAssembler(inputCols=["year"], outputCol="features")
new_df = assembler.transform(new_df)

prediction = model.transform(new_df)
prediction.select("year", "prediction").show()

+----+------------------+
|year|        prediction|
+----+------------------+
|2023|175.16564294006457|
+----+------------------+

